In [ ]:
import pandas as pd
from IPython.display import display
from src.research_config import ResearchConfig
from src.fractional_OU import filter_antipersistent_pairs
from src.fractional_OU import compute_structural_t70
from src.fractional_OU import select_top_pairs_by_structural_t70


# 03 Pair Eligibility

Filter stable anti-persistent fits, estimate structural convergence horizons and select up to 40 pairs. A smaller valid population is reported without loosening thresholds.

Run cells from top to bottom, or choose **Run All** for this notebook only. Each module saves its outputs for the next notebook. Restart the kernel after pulling code changes.


## 1. Settings


In [ ]:
cfg = ResearchConfig().validate()


## 2. Anti persistent and stable fits

Require 0 < H < 0.5, positive sigma/variance and daily Euler stability 0 < kappa < 2.


In [ ]:
fou = pd.read_parquet("fractional_ou_parameters.parquet")
pool = filter_antipersistent_pairs(fou)
print(f"{len(pool)} of {len(fou)} fits pass model eligibility")
if pool.empty:
    raise ValueError("No stable anti-persistent fits.")
display(pool.head())


## 3. Structural convergence horizons

This is the expensive formation simulation. Its target, path count and seed are the shared defaults in src/research_config.py.


In [ ]:
structural = compute_structural_t70(
    pool,
    starting_z=cfg.entry_z,
    target_probability=cfg.target_probability,
    max_horizon_days=cfg.structural_horizon,
    n_paths=cfg.n_paths,
    seed=cfg.seed,
)
structural.to_parquet("structural_results.parquet")
display(structural[["pair", "structural_t70", "structural_probability_max"]].head(15))


## 4. Rank and save the portfolio

The full finite-horizon pool is saved separately for matched placebo sampling.


In [ ]:
eligible_pool = structural.loc[structural.structural_t70.notna()].copy()
top_pairs = select_top_pairs_by_structural_t70(eligible_pool, cfg.top_n)
eligible_pool.to_parquet("eligible_pool.parquet")
top_pairs.to_parquet("eligible_pairs.parquet")
if top_pairs.empty:
    raise ValueError("No finite structural horizons.")
top_pairs.structural_t70.describe().to_frame("horizon").to_parquet(
    "selected_horizon_summary.parquet"
)
print(f"{len(top_pairs)} selected pairs from {len(eligible_pool)} eligible pairs")
display(top_pairs[["pair", "hurst", "structural_t70", "structural_probability_max"]])
